# DizerCore LoRA Training — auto-matches installed model

Train a TrinityCore investigation-prompt LoRA adapter on Kaggle's free T4 GPU.

**Setup:**

1. Create a Kaggle dataset and upload `dizercore-dataset.jsonl` from the Pi
   (`/data/training/`). Copy its input path into `DATASET` below.
2. Attach a T4 GPU (Settings → Accelerator → T4 x2).
3. Run → Save & Run All. Interrupting the session kills the run —
   Save & Run All runs it server-side and survives.

**Output:** `dizercore-q4_k_m.gguf` appears in the Output panel when done —
upload it via the Web UI Training tab (or `training-deploy.sh` on the Pi).

**Realistic runtime:** ~18h for a full 57k-example epoch — longer than the
12h session limit. Checkpoints save every 200 steps to
`/kaggle/working/dizercore/checkpoints`. Before the session dies, save that
dir as a Kaggle dataset; next session attach it and set `PREV_CHECKPOINTS`
to resume where it left off.

**Base model:** read from the dataset meta row (line 1) so the adapter
always matches whatever model the Pi has installed. No manual edits needed
when you switch models.

In [ ]:
import subprocess, sys, os, gc, json, shutil

# ============ INSTALL ============
print('=== Install ===', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'transformers', 'peft', 'bitsandbytes',
                'accelerate', 'datasets'], check=True)

os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'

# ============ CONFIG ============
DATASET = '/kaggle/input/datasets/vekzla/dizercore-wow/dizercore-dataset.jsonl'
# If you saved checkpoints as a Kaggle dataset last session, put its path
# here (e.g. '/kaggle/input/dizercore-checkpoints/checkpoints'). Otherwise
# leave as None.
PREV_CHECKPOINTS = None
WORK = '/kaggle/working/dizercore'
ADAPTER = f'{WORK}/adapter'
MERGED = f'{WORK}/merged'
GGUF_F16 = f'{WORK}/dizercore-f16.gguf'
GGUF_Q4 = f'{WORK}/dizercore-q4_k_m.gguf'
CKPT_DIR = f'{WORK}/checkpoints'
os.makedirs(WORK, exist_ok=True)

assert os.path.isfile(DATASET), f'Dataset not found: {DATASET}'

# Base model comes from the dataset meta row (line 1, written by
# dataset-builder.py on the Pi). Falls back to 3B-Instruct.
BASE = 'Qwen/Qwen2.5-3B-Instruct'
try:
    with open(DATASET) as _f:
        BASE = json.loads(_f.readline()).get('base_model', BASE)
except Exception as _e:
    print(f'No base_model tag in dataset ({_e}); using default')
print('Base model:', BASE)
print('Dataset:', DATASET)

# ============ RESTORE CHECKPOINTS (resume across sessions) ============
if PREV_CHECKPOINTS and os.path.isdir(PREV_CHECKPOINTS):
    shutil.copytree(PREV_CHECKPOINTS, CKPT_DIR, dirs_exist_ok=True)
    print(f'Restored checkpoints from {PREV_CHECKPOINTS}', flush=True)

# ============ LOAD ============
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
print('=== Load dataset ===')
ds = load_dataset('json', data_files=DATASET, split='train')
ds = ds.filter(lambda x: 'text' in x)
print(f'{len(ds)} training examples (meta row excluded)')

def fmt(ex):
    msgs = ex['messages']
    text = ''.join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in msgs)
    return {'text': text}
ds = ds.map(fmt, remove_columns=ds.column_names)
print(ds[0]['text'][:500])

print('=== Load base model (4-bit) ===')
tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True,
                         bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(BASE,
                                             quantization_config=bnb,
                                             device_map='auto',
                                             torch_dtype=torch.float16)
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                  'gate_proj', 'up_proj', 'down_proj'],
                  bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()

# ============ TOKENIZE ============
print('=== Tokenizing ===', flush=True)
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Measure every example first so nothing gets truncated.
def measure(ex):
    return {'n': len(tokenizer(ex['text'])['input_ids'])}

ds_len = ds.map(measure, remove_columns=ds.column_names, num_proc=2)
MAX_LEN = max(ds_len['n'])
print(f'Longest example: {MAX_LEN} tokens')
del ds_len; gc.collect()

# Auto batch sizing: long sequences OOM at batch 8 on a 15GB T4.
BATCH, ACCUM = (4, 4) if MAX_LEN > 2048 else (8, 2)
print(f'Batch: {BATCH} x accum {ACCUM} (effective {BATCH * ACCUM})', flush=True)

def tokenize(ex):
    return tokenizer(ex['text'], truncation=True, max_length=MAX_LEN)

ds = ds.map(tokenize, remove_columns=ds.column_names)
print(f'Tokenized {len(ds)} examples, max_length={MAX_LEN}')

# ============ TRAIN ============
print('=== Train ===', flush=True)
args = TrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    save_only_model=False,
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)

ckpt = None
if os.path.isdir(CKPT_DIR):
    cps = [os.path.join(CKPT_DIR, d) for d in os.listdir(CKPT_DIR)
           if d.startswith('checkpoint-') and d.rsplit('-', 1)[-1].isdigit()]
    if cps:
        ckpt = max(cps, key=lambda p: int(p.rsplit('-', 1)[-1]))
        print(f'Resuming from {ckpt}', flush=True)
trainer.train(resume_from_checkpoint=ckpt)
trainer.save_model(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print(f'Adapter saved: {ADAPTER}')

# ============ MERGE ============
print('=== Merge adapter into base model ===', flush=True)
del model, trainer
gc.collect()
torch.cuda.empty_cache()

from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE,
                                            device_map='auto',
                                            torch_dtype=torch.float16)
merged = PeftModel.from_pretrained(base, ADAPTER)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED)
tokenizer.save_pretrained(MERGED)
del base, merged
gc.collect()
torch.cuda.empty_cache()
print(f'Merged model: {MERGED}')

# ============ CONVERT TO GGUF ============
print('=== Convert to GGUF ===', flush=True)
LLAMA_DIR = '/kaggle/working/llama.cpp'
if not os.path.isdir(LLAMA_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggml-org/llama.cpp.git', LLAMA_DIR],
                   check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', f'{LLAMA_DIR}/requirements.txt'], check=True)

subprocess.run(['cmake', '-B', 'build',
                '-DGGML_NATIVE=OFF', '-DLLAMA_CURL=OFF'],
               cwd=LLAMA_DIR, check=True)
subprocess.run(['cmake', '--build', 'build', '--config', 'Release',
                '-j', '2', '--target', 'llama-quantize'],
               cwd=LLAMA_DIR, check=True)

subprocess.run([sys.executable, f'{LLAMA_DIR}/convert_hf_to_gguf.py',
                MERGED, '--outfile', GGUF_F16, '--outtype', 'f16'], check=True)

print('=== Quantizing to Q4_K_M ===', flush=True)
subprocess.run([f'{LLAMA_DIR}/build/bin/llama-quantize',
                GGUF_F16, GGUF_Q4, 'Q4_K_M'], check=True)

# ============ DONE ============
size_mb = os.path.getsize(GGUF_Q4) / 1024 / 1024
print('=== DONE ===', flush=True)
print(f'Base trained: {BASE}', flush=True)
print(f'GGUF: {GGUF_Q4} ({size_mb:.0f} MB)', flush=True)
print('IMPORTANT: before the session ends, also save the checkpoints dir as a')
print('Kaggle dataset (Output -> New Dataset on /kaggle/working/dizercore/checkpoints)')
print('so the next session can resume. Set PREV_CHECKPOINTS to that path.', flush=True)
print('Then download dizercore-q4_k_m.gguf and upload via the Web UI Training tab.',
      flush=True)
if not os.path.isdir('/kaggle'):
    from IPython.display import FileLink, display
    display(FileLink(GGUF_Q4, result_html_prefix='Download: '))